# 01 — Bronze: new files and corrected tickets

Read the existing volume and maintain one row per Id+LocationId+BusinessDate. The manifest detects new/changed files by size and modification time. Incoming tickets replace their complete versions, including arrays. Unchanged repeats do not change data. Duplicate keys are not resolved arbitrarily.

Omitted tickets are retained by default. Enable `replace_files` only for business-confirmed complete replacement files. Dimensions are complete catalogs. See docs/operacion.md for contracts, adoption, failures and limitations.

In [ ]:
from pyspark.sql import functions as F
from pipeline_support import load_config, PipelineContext, assert_key, assert_same_keys
CFG = load_config()
CATALOG = CFG["catalog"]
SCHEMA = CFG["schemas"]["bronze"]
K = ["CheckId", "LocationId", "BusinessDate"]
EXPECTED_TICKETS = CFG["original"]["tickets"]
EXPECTED_DAYS = CFG["original"]["days"]
TOLERANCE = float(CFG["tolerance"])
RUN_ORIGINAL_REGRESSION = CFG["run_original_regression"]
ctx = PipelineContext(spark, CFG, "bronze")
ctx.begin()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
def table(name): return f"{CATALOG}.{SCHEMA}.{name}"
def save(df,name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table(name))
    return spark.table(table(name))
print("Destino:", CATALOG+"."+SCHEMA, "Run:", ctx.run_id)
from pipeline_support import list_json_files, ingest_source, canonical_path
BASE = CFG["base_path"]
root = f"{BASE}/facts/checks/brand=rmh"
inventory = list_json_files(dbutils, root)
paths = {"locations":"dimensions/locations/locations.json", "items":"dimensions/items/items.json", "revenue_centers":"dimensions/revenue_centers/revenueCenters.json", "employees":"dimensions/employees/employees.json"}
dimension_inventory = {}
for name, path in paths.items():
    files = dbutils.fs.ls(f"{BASE}/{path}")
    assert len(files) == 1 and not files[0].isDir(), f"Catálogo ausente o ambiguo: {name}"
    item = files[0]
    dimension_inventory[name] = {canonical_path(item.path): (int(item.size), int(item.modificationTime))}
all_paths = set(inventory).union(*(set(x) for x in dimension_inventory.values()))
assert set(map(canonical_path, CFG["force_files"])) <= all_paths, "force_files contiene rutas inexistentes"


## Synchronize tickets and validate coverage

Only new/changed files are read. Each input is frozen in a Delta staging table before MERGE. Duplicate keys and invalid dates stop the batch. Historical counts apply only in original-regression mode.

In [ ]:
ingest_source(ctx, dbutils, "checks", inventory, ["Id", "LocationId", "BusinessDate"])
raw = spark.table(table("checks_raw"))
assert_key(raw, ["Id", "LocationId", "BusinessDate"], "checks_raw")
coverage = raw.agg(F.count("*").alias("tickets"),F.countDistinct("BusinessDate").alias("days"),F.countDistinct("source_file").alias("files"),F.min("BusinessDate").alias("first"),F.max("BusinessDate").alias("last")).first()
assert coverage.tickets > 0 and coverage.days > 0, "Origen vacío; revisar antes de publicar"
if RUN_ORIGINAL_REGRESSION:
    expected = CFG["original"]
    assert coverage.tickets == expected["tickets"] and coverage.days == expected["days"] and coverage.files == expected["files"], f"Regresión original: {coverage}"
    assert str(coverage.first) == expected["first_date"] and str(coverage.last) == expected["last_date"], "Regresión original: periodo distinto"
print("Tickets:",coverage.tickets,"Fechas:",coverage.days,"Archivos:",coverage.files)
display(raw.select("Id","LocationId","BusinessDate","source_file").limit(10))


## Synchronize dimensions and complete Bronze

Each catalog is a complete snapshot keyed by Id. Invalid references are retained for Silver checks. The manifest advances only after each merge. Failures leave Bronze RUNNING and block Silver until recovery.

In [ ]:
for name in paths:
    ingest_source(ctx, dbutils, name, dimension_inventory[name], ["Id"])
# Do not approve a batch if the source changed during processing.
assert list_json_files(dbutils, root) == inventory, "Cambió el inventario de tickets durante el lote"
for name, path in paths.items():
    item = dbutils.fs.ls(f"{BASE}/{path}")
    observed = {canonical_path(x.path): (int(x.size), int(x.modificationTime)) for x in item}
    assert observed == dimension_inventory[name], f"Cambió el catálogo {name} durante el lote"
ctx.finish([table(name+"_raw") for name in ["checks", *paths]])
